# 📝 추출 품질 과제 LV1: 지표를 하나씩 계산하기

교안 01의 절 순서를 그대로 따라 의료 논문 추출을 평가합니다.  
**스키마 준수율 → 근거 원문 일치율 → 샘플 정밀도 → 골드 작성 → 완전일치 P/R/F1** 순서입니다.  
계약 정의, 구조화 추출, 이름 조회는 지난 시간의 내용입니다. 이번에는 제공된 추출 결과를 읽고 무엇이 맞거나 틀렸는지 설명합니다.

- 코딩 9문항과 서술형 2문항입니다. 답안은 위에서부터 작성합니다.  
- 데이터는 `lv1_corpus.jsonl` 여섯 문서와 `lv1_triples.jsonl` 31행입니다. 이번 과제에는 실제 모델 호출이 없습니다.  
- 작은 골드, 검증 기록, 평가 결과를 `output/lv1_*.json`에 남기면 완료입니다.  
- 변수에 담는 **모든 비율은 반올림하지 않은 나눗셈 결과**입니다. 자릿수를 줄이는 것은 화면 출력에서만 합니다.  
- 사람이 판정한 **샘플 정밀도**와 골드의 **완전일치 P/R/F1**은 서로 다른 값입니다. 한 숫자로 합치지 않습니다.

In [ ]:
# [제공 코드]

# 실습에 공통으로 쓸 파일 경로와 읽기, 저장 함수를 준비합니다.

import json
import random
from collections import Counter
from pathlib import Path

data_dir = Path("data")  # 제공된 원문, 추출된 트리플, 골드 파일이 있는 폴더입니다.
output_dir = Path("output")  # 직접 계산한 지표와 검토 기록을 저장할 폴더입니다.
output_dir.mkdir(exist_ok=True)

def load_rows(filename):
    """data 폴더의 JSONL 파일을 딕셔너리 목록으로 읽습니다."""
    return [json.loads(line) for line in (data_dir / filename).read_text(encoding="utf-8").splitlines() if line.strip()]

def write_json(filename, value):
    """이번 실습의 결과를 output 폴더에 JSON으로 저장합니다."""
    (output_dir / filename).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def write_rows(filename, rows):
    """검토 기록을 한 줄에 한 항목인 JSONL로 저장합니다."""
    content = "\n".join(json.dumps(row, ensure_ascii=False) for row in rows)
    (output_dir / filename).write_text(content + "\n", encoding="utf-8")

def triple_key(row):
    """고유 관계를 비교할 (주어, 관계, 목적어) 튜플을 돌려줍니다."""

    # 평가 전에 표기를 바꾸지 않습니다. 이름 정규화는 다음 단원에서 배웁니다.
    return (row["subject"], row["relation"], row["object"])

In [ ]:
# [제공 코드]

# 논문 추출을 관계, 타입 규칙에 따라 통과와 기각으로 나눌 함수를 준비합니다.

signatures = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

def check_signature(row):
    """관계, 타입 위반 사유를 돌려주고, 통과하면 None을 돌려줍니다."""
    if row["relation"] not in signatures:
        return "허용 관계가 아님"
    # signatures의 값은 해당 관계가 요구하는 주어 타입과 목적어 타입입니다.
    subject_type, object_type = signatures[row["relation"]]
    if row["subject_type"] != subject_type:
        return f"주어 타입이 {subject_type} 이어야 함"
    if row["object_type"] != object_type:
        return f"목적어 타입이 {object_type} 이어야 함"

    return None

def split_schema(rows):
    """추출 목록을 스키마 통과 목록과 사유가 붙은 기각 목록으로 나눕니다."""
    valid, rejected = [], []
    for row in rows:
        reason = check_signature(row)
        if reason is None:
            valid.append(row)
        else:
            # 원본은 유지하고 기각 목록에만 사유를 덧붙입니다.
            rejected.append(dict(row, reject_reason=reason))

    return valid, rejected

In [ ]:
# [제공 코드]

# 추출과 골드를 비교해 TP, FP, FN, 정밀도, 재현율, F1을 계산할 함수를 정의합니다.

def measure_exact(rows, gold_rows):
    """고유 관계의 완전일치 TP, FP, FN과 정밀도, 재현율, F1을 돌려줍니다."""
    # 집합으로 바꿔 같은 관계를 여러 번 뽑아도 한 번만 셉니다.
    predicted = {triple_key(row) for row in rows}
    expected = {triple_key(row) for row in gold_rows}
    tp = len(predicted & expected)  # 추출 결과와 골드 양쪽에 있는 관계입니다.
    fp = len(predicted - expected)  # 골드에 없는 추출입니다. 표기 차이도 포함합니다.
    fn = len(expected - predicted)  # 골드에는 있지만 추출하지 못한 관계입니다.
    # 분모가 없으면 0점 대신 미산출(None)로 남깁니다.
    precision = tp / len(predicted) if predicted else None
    recall = tp / len(expected) if expected else None

    # 골드가 있는데 아무것도 뽑지 않으면 F1은 0입니다. 골드가 없으면 평가에서 별도 표시합니다.
    f1 = 2 * tp / (2 * tp + fp + fn) if expected else None

    return {"predicted": len(predicted), "gold": len(expected), "tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}

In [ ]:
# [제공 코드] 어제 만든 스키마 검사 결과를 받아 오늘의 평가를 시작합니다.
corpus = load_rows("lv1_corpus.jsonl")
docs = {row["doc_id"]: row for row in corpus}
triples = load_rows("lv1_triples.jsonl")
valid, rejected = split_schema(triples)
print(f"원문 {len(docs)}편 · 추출 {len(triples)}행 · 스키마 통과 {len(valid)}행")

**이번 평가의 기준**

- 평가 대상은 지정한 문서와 관계 범위입니다. 범위 밖 결과는 먼저 분리합니다.  
- 한 항목은 고유한 **(주어, 관계, 목적어)**입니다. 같은 관계가 여러 문서에 있어도 한 번만 셉니다. 출처와 근거는 별도 기록으로 보존합니다.  
- 세 값이 **문자열까지 모두 같을 때** 일치로 셉니다. 대소문자·괄호가 다르면 다른 항목입니다.  
- 이 점수는 **저장된 골드와의 완전일치**입니다. FP라고 해서 반드시 원문의 의미를 틀리게 읽었다는 뜻은 아닙니다. 원문 근거와 표기 차이를 함께 확인합니다.  
- 스키마 검사, 근거 원문 일치 검사, 사람이 판정한 근거 적합 여부는 서로 다른 검사입니다. 각각의 대상과 분모를 적습니다.

## 1. 스키마 준수율의 분모를 확인하세요

**배경**: 지난 시간의 검사를 새로 만들지 않고, 제공된 통과·기각 목록에서 검사 대상과 분모를 확인합니다.

**요구사항**

- **`schema_report`**를 딕셔너리로 만드세요. 정수 **`raw`**, **`valid`**, **`rejected`**에는 각 목록의 길이를 넣습니다.  
- 실수 **`rate`**에는 `스키마 통과 행 수 / 추출 전체 행 수`를 넣습니다. 반올림하지 않습니다.  
- 딕셔너리 **`reasons`**에는 `rejected`의 **`reject_reason`**별 건수를 넣으세요. Counter 결과를 dict로 바꾸면 됩니다.

**확인 기준**: raw=31, valid=28, rejected=3입니다. rate의 분모는 31이고 reasons 값의 합은 3입니다.

<details><summary>힌트</summary>

**접근방법**: 스키마 검사의 행 수와 사유를 한곳에 모읍니다.

**세부구현**

1. 중복 제거나 근거 검사로 분모를 바꾸지 마세요.  
2. 기각 사유는 제공된 reject_reason에서 읽으세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert schema_report["raw"] == len(triples) == 31, "raw에는 중복을 제거하지 않은 추출 31행을 넣으세요."
assert schema_report["valid"] == len(valid) == 28 and schema_report["rejected"] == len(rejected) == 3, "valid 28행과 rejected 3행을 각각 넣으세요."
assert schema_report["rate"] == len(valid) / len(triples),             "rate는 반올림하지 않은 나눗셈 결과이고, 분모는 기각 3행을 포함한 전체 행 수입니다."
assert schema_report["reasons"] == dict(Counter(row["reject_reason"] for row in rejected)), "reasons는 reject_reason별 건수를 담은 dict입니다."
assert schema_report["valid"] + schema_report["rejected"] == schema_report["raw"], "통과와 기각을 더하면 전체 행 수가 되어야 합니다."
print("✅ 스키마 준수율 통과!")

## 2. 근거 원문 일치율을 계산하고 검토 대상을 남기세요

**배경**: 문자열이 원문에 없다는 것만으로 관계 전체가 거짓이라고 단정할 수 없습니다. 줄임표나 문장부호 차이도 사람이 확인합니다.

**요구사항**

- **`evidence_status(row)`** 함수를 만드세요. 반환값은 문자열입니다.  
- evidence가 공백뿐이면 `"근거 없음"`입니다. 그렇지 않으면 근거 문자열을 공백까지 그대로 둔 채 해당 출처 원문과 대조해, 전체가 들어 있으면 `"원문 일치"`, 아니면 `"원문 확인 필요"`입니다.  
- 해당 원문은 **`docs[row["source_doc_id"]]["text"]`**입니다. 다른 문서를 검색하지 않습니다.  
- **`evidence_ok`**를 `valid` 중 `"원문 일치"`인 원본 트리플의 리스트로 만드세요. 원래 순서를 유지합니다.  
- **`evidence_review`**를 `valid` 중 `"원문 일치"`가 아닌 원본 트리플의 리스트로 만드세요. 기존 `valid`를 수정하지 않습니다.  
- **`evidence_report`**를 딕셔너리로 만드세요. 정수 **`checked`**는 근거를 검사한 행 수, 정수 **`verbatim`**은 원문 일치 행 수, 실수 **`rate`**는 `verbatim / checked`, 정수 **`review`**는 검토 대상 행 수입니다.  
- 각 검토 대상의 트리플 키, 출처, 근거를 출력하세요.

**확인 기준**: checked=28, verbatim=22, review=6이고 rate는 22/28입니다. 스키마 통과 목록 28행은 그대로 유지되며 검토 대상을 자동으로 삭제하지 않습니다.

<details><summary>힌트</summary>

**접근방법**: 빈 근거와 원문 불일치를 구분한 뒤 통과 목록과 검토 목록을 각각 만듭니다.

**세부구현**

1. strip은 빈 문자열을 판정할 때 사용하세요.  
2. 원문 일치 여부는 근거 전체를 in으로 비교하세요.  
3. 원본 목록은 유지하고 새 리스트 두 개를 만드세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(valid) == 28, "검토 대상을 valid에서 삭제하지 마세요."
assert evidence_ok == [row for row in valid if evidence_status(row) == "원문 일치"], "원본 순서를 유지한 채 원문 일치 행만 담으세요."
assert evidence_review == [row for row in valid if evidence_status(row) != "원문 일치"], "근거 없음과 원문 확인 필요를 함께 검토 목록에 담으세요."
assert len(evidence_ok) == 22 and len(evidence_review) == 6, "22행 통과, 6행 검토 대상입니다."
assert evidence_report == {"checked": 28, "verbatim": 22, "rate": 22 / 28, "review": 6},             "네 값의 키 이름과 순서를 확인하고, rate는 반올림하지 말고 그대로 넣으세요."
probe = dict(valid[0], evidence="   ")
assert evidence_status(probe) == "근거 없음", "빈 근거를 따로 구분하세요."
probe["evidence"] = docs[probe["source_doc_id"]]["sentences"][0]
assert evidence_status(probe) == "원문 일치", "출처 원문에 그대로 있는 근거는 원문 일치입니다."
probe["evidence"] = "이 문장은 출처에 없는 시험용 근거입니다."
assert evidence_status(probe) == "원문 확인 필요", "불일치를 최종 오답으로 단정하지 않습니다."
other_doc_id = next(doc_id for doc_id in docs if doc_id != probe["source_doc_id"])
probe["evidence"] = docs[other_doc_id]["sentences"][0]
assert evidence_status(probe) == "원문 확인 필요", "다른 문서까지 검색하면 엉뚱한 근거를 통과시킵니다."
print("✅ 근거 원문 일치율 통과!")

In [ ]:
# [제공 코드] 기존 사람 채점 정보를 유효 트리플 28건 모두에 명시합니다.
# score는 이 추출본의 근거, 개체 범위에 대한 판정입니다. 완전일치 TP와 다릅니다.
# 다른 추출본의 근거가 달라지면 같은 트리플 이름이어도 다시 읽고 채점해야 합니다.
scores = {
    ("Crohn's disease", "ASSOCIATES", "NOD2"): 1,
    ("Crohn's disease", "ASSOCIATES", "ATG16L1"): 1,
    ("ADHD", "ASSOCIATES", "CACNA1C"): 1,
    ("ADHD", "ASSOCIATES", "CACNA1D"): 1,
    ("ADHD", "ASSOCIATES", "CACNB1"): 1,
    ("ADHD", "ASSOCIATES", "CACNA2D3"): 1,
    ("AML", "BINDS", "CACNA1C"): 1,
    ("AML", "BINDS", "CACNA1D"): 1,
    ("AML", "BINDS", "CACNB1"): 1,
    ("AML", "BINDS", "CACNA2D3"): 1,
    ("tirzepatide", "BINDS", "GLP-1 receptor"): 1,
    ("tirzepatide", "BINDS", "GIP receptor"): 1,
    ("retatrutide", "BINDS", "GLP-1 receptor"): 1,
    ("retatrutide", "BINDS", "GIP receptor"): 1,
    ("retatrutide", "BINDS", "glucagon (GCG) receptors"): 1,
    ("semaglutide", "TREATS", "renal injury"): 0,
    ("tirzepatide", "TREATS", "renal injury"): 0,
    ("retatrutide", "TREATS", "renal injury"): 0,
    ("semaglutide", "TREATS", "renal function impairment"): 1,
    ("tirzepatide", "TREATS", "renal function impairment"): 1,
    ("retatrutide", "TREATS", "renal function impairment"): 1,
    ("unilateral ureteral obstruction (UUO)", "PRESENTS", "renal swelling"): 1,
    ("unilateral ureteral obstruction (UUO)", "PRESENTS", "pathological alterations"): 0,
    ("PPIs", "BINDS", "H+/K+-ATPase"): 0,
    ("PPIs", "TREATS", "gastroesophageal reflux disease"): 0,
    ("PPIs", "TREATS", "peptic ulcer disease"): 0,
    ("PPIs", "TREATS", "stress-related mucosal lesions"): 0,
    ("PPIs", "TREATS", "Helicobacter pylori infection"): 0,
}

**사람 채점표의 범위**

위 표는 이 추출본의 유효 28행을 모두 읽고 매긴 기존 채점 기록입니다. 근거가 관계를 뒷받침하고, 이름이 허용된 개체 범위에 속하면 1점입니다. 예를 들어 `PPIs`는 약물 한 개가 아닌 약효 분류이므로 이 추출 과제의 `Compound`로 인정하지 않았습니다.

교안 01의 3절대로 **샘플 정밀도의 모집단은 스키마와 근거 원문 검사를 모두 통과한 목록**입니다. 채점표는 28행 전체를 덮지만, 아래 두 문제는 두 검사를 통과한 22행만 대상으로 계산합니다. 표에 없는 항목은 **미채점**이며 1점이나 0점으로 대신하지 않습니다.

## 3. 미채점을 차단하고 판정 점수의 평균을 구하세요

**배경**: 채점표에 없는 항목을 정답으로 취급하면 읽지 않은 결과가 만점이 됩니다.

**요구사항**

- **`judged_precision(rows, score_table)`** 함수를 만드세요. 원본 트리플 리스트와 키에서 0 또는 1로 가는 딕셔너리를 받습니다.  
- 입력의 트리플 키가 채점표에 하나라도 없으면 **`ValueError`**를 발생시키세요. 기본 점수를 사용하지 않습니다.  
- 모든 입력이 채점됐으면 점수 합계를 행 수로 나눈 실수를, 빈 목록이면 **`None`**을 반환하세요.  
- **`precision_all`**에 두 검사를 통과한 `evidence_ok` 전체의 값을 저장하세요.

**확인 기준**: 두 검사를 통과한 22행 중 16행이 1점이므로 precision_all은 16/22입니다. 같은 함수에 28행짜리 valid를 넣으면 19/28이 나옵니다. 모집단이 달라지면 값도 달라집니다. 미채점 항목을 넣으면 ValueError가 발생하고 빈 목록은 None입니다.

<details><summary>힌트</summary>

**접근방법**: 점수를 더하기 전에 채점표가 입력 전체를 덮는지 확인합니다.

**세부구현**

1. triple_key로 입력 키 집합을 만드세요.  
2. score_table의 키 집합과 차집합을 구하세요.  
3. 누락이 없을 때 대괄호로 점수를 읽어 평균을 내세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert set(scores) == {triple_key(row) for row in valid}, "이 표는 유효 목록 전체의 명시적 채점표입니다."
assert set(scores.values()) <= {0, 1}, "채점표의 값은 0 또는 1입니다."
assert abs(precision_all - 16 / 22) < 1e-12, "모집단은 두 검사를 통과한 22행입니다."
assert abs(judged_precision(valid, scores) - 19 / 28) < 1e-12, "모집단을 바꾸면 값도 달라집니다."
assert judged_precision([], scores) is None, "분모가 0이면 비율을 만들지 않습니다."
assert judged_precision(evidence_ok[:2], scores) == 1.0, "두 행이 모두 1점이면 평균은 1.0입니다."
probe_scores = dict(scores)
probe_scores.pop(triple_key(evidence_ok[0]))
try:
    judged_precision(evidence_ok[:1], probe_scores)
except ValueError:
    pass
except Exception as error:
    raise AssertionError(f"미채점은 ValueError로 알려야 합니다. 지금은 {type(error).__name__}가 났습니다.")
else:
    raise AssertionError("미채점 항목은 기본 점수 대신 ValueError로 알려야 합니다.")
print("✅ 미채점 차단·판정 정밀도 통과!")

## 4. 두 검사를 통과한 목록에서 표본을 뽑아 샘플 정밀도를 구하세요

**배경**: 실제 작업에서는 전체를 다 읽을 수 없어 표본만 판정합니다. 전수 채점표가 있는 이 자료로 표본 추정과 전수 측정의 차이를 확인합니다.

**요구사항**

- **`sample`**을 `random.Random(42)` 객체의 sample 메서드로 **`evidence_ok`**에서 뽑은 10행 리스트로 만드세요. evidence_ok의 순서를 바꾸지 않습니다.  
- **`sample_keys`**는 뽑은 행의 triple_key를 뽑힌 순서대로 모은 리스트입니다.  
- **`sample_precision`**을 앞 문제의 `judged_precision`으로 계산하세요.  
- 표본과 전수의 분모, 비율을 함께 출력하세요. 이것은 같은 추출본 안의 비교이며 개선판의 성능 비교가 아닙니다.

**확인 기준**: 표본은 10행이고 키는 중복되지 않습니다. sample_precision은 0.80이고 전수 22행은 약 0.727입니다. 스키마만 통과한 6행은 표본 후보에 들어오지 않습니다.

<details><summary>힌트</summary>

**접근방법**: 표본 선택용 난수 객체를 만들고 선택한 항목의 키까지 기록합니다.

**세부구현**

1. Random 객체에 시드 42를 전달하세요.  
2. 제공 순서의 evidence_ok에서 10행을 선택하세요.  
3. 앞 문제의 같은 함수로 평균을 구하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert sample == random.Random(42).sample(evidence_ok, 10), "시드와 evidence_ok 순서를 확인하세요."
assert sample_keys == [triple_key(row) for row in sample], "정렬하지 말고 뽑힌 순서 그대로 담으세요."
assert len(set(sample_keys)) == 10, "같은 관계가 두 번 들어갔는지 확인하세요."
assert all(row in evidence_ok for row in sample), "표본은 두 검사를 통과한 목록에서만 뽑습니다."
assert sample_precision == judged_precision(sample, scores), "앞 문제의 함수로 계산하세요."
assert abs(sample_precision - 0.80) < 1e-12, "표본 10행 중 8행이 1점입니다."
print("✅ 샘플 정밀도 통과!")

## 5. 자동 검사와 사람 판정의 차이를 설명하세요

**배경**: 스키마를 통과해도 근거가 그 관계를 뒷받침하지 못할 수 있습니다.

**요구사항**

- **판정 기록**을 세 문장 이상 작성하세요. 대상은 `(semaglutide, TREATS, renal injury)`입니다.  
- `valid`에서 이 항목의 evidence를 읽고, 원문 `docs["PMC13495082"]["sentences"][2]`와 비교하세요.  
- 스키마 검사 결과, 근거 원문 일치 검사 결과, 사람 점수를 각각 적고 0점 이유를 설명하세요. 근거에 언급된 약물 이름과 `therapeutic potential` 표현을 사용하세요.  
- 이 항목이 표본에 들어오지 않은 이유와, 그래서 샘플 정밀도만으로는 이 오류를 못 찾는다는 점을 적으세요.

**확인 기준**: 스키마, 근거 원문 일치, 사람 판정 세 검사를 구분하고 배경지식 대신 원문 표현을 이유로 사용합니다. 자동 채점은 하지 않습니다. 작성한 뒤 정답 노트북의 모범 서술과 비교하세요.

<details><summary>힌트</summary>

**접근방법**: 원문 구절, 판정 기준, 결론을 순서대로 연결하세요.

**세부구현**

1. 판단에 사용한 원문 표현을 짧게 인용하세요.  
2. 그 표현이 어떤 판정 기준을 충족하거나 충족하지 못하는지 설명하세요.  
3. 점수의 문제와 추출 내용의 문제를 구분하세요.

</details>

*(여기에 원문 근거와 판단을 작성하세요.)*

In [ ]:
# [제공 코드] 작은 골드는 이 문서의 두 문장만 대상으로 직접 작성합니다.
mini_doc_id = "PMC13496164"
mini_sentences = {i: docs[mini_doc_id]["sentences"][i] for i in (12, 13)}
for sent_id, sentence in mini_sentences.items():
    print(f"문장 {sent_id}: {sentence}")

## 6. 평가 범위를 기록하세요

**배경**: 한 문서의 두 문장을 주석했다고 해서 다른 문서까지 평가할 수는 없습니다. 작은 골드의 범위부터 기록합니다.

**요구사항**

- **`mini_plan`**을 딕셔너리로 만드세요.  
- **`doc_ids`**는 `[mini_doc_id]`, **`sent_ids`**는 `[12, 13]`, **`relations`**는 `["ASSOCIATES"]`인 리스트입니다.  
- **`unit`**은 `"unique_triple"`, **`matching`**은 `"exact"`인 문자열입니다. 위 평가 기준 상자의 두 규칙, 곧 한 항목은 고유한 세 값이고 문자열까지 모두 같아야 한다는 규칙을 적는 자리입니다.  
- **`criterion`**은 질병과 유전자의 연관을 원문에서 확인하고, 같은 문장에 이름만 함께 나온 것은 제외한다는 기준을 본인 문장으로 적으세요. 무엇을 빼는지 분명해지도록 문장에 **제외**라는 말을 넣습니다.  
- 이름은 원문 표기를 유지합니다. 이번 작은 골드는 질병에서 유전자로 가는 방향만 담습니다.

**확인 기준**: 문서, 문장, 관계 범위와 완전일치 규칙이 모두 들어 있습니다. criterion은 한 문장 이상이며 관계를 인정할 조건과 제외할 조건을 함께 설명합니다.

<details><summary>힌트</summary>

**접근방법**: 평가가 끝날 때까지 고정할 조건을 딕셔너리 하나에 모읍니다.

**세부구현**

1. 목록인 값과 문자열인 값을 구분하세요.  
2. 기준 문장에는 관계를 포함할 조건과 제외할 조건을 적으세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert mini_plan["doc_ids"] == [mini_doc_id], "준비 셀의 문서 한 편만 대상으로 합니다."
assert mini_plan["sent_ids"] == [12, 13], "주석할 두 문장 번호를 기록하세요."
assert mini_plan["relations"] == ["ASSOCIATES"], "질병과 유전자의 연관만 대상으로 합니다."
assert mini_plan["unit"] == "unique_triple" and mini_plan["matching"] == "exact", "unit과 matching의 값을 지문 그대로 넣으세요."
assert isinstance(mini_plan["criterion"], str) and len(mini_plan["criterion"].strip()) >= 20,             "criterion을 한 문장 이상으로 적으세요."
assert "제외" in mini_plan["criterion"], "포함할 조건과 함께 무엇을 제외하는지도 적으세요."
print("✅ 범위 기록 통과. 판정 기준의 뜻은 원문과 함께 직접 확인하세요.")

## 7. 두 문장을 읽고 골드를 직접 작성하세요

**배경**: 저장된 정답을 조회하는 활동과 원문에서 정답을 만드는 활동은 다릅니다. 이번에는 위에 출력한 두 문장을 직접 읽습니다.

**요구사항**

- **`mini_gold`**를 딕셔너리 리스트로 직접 작성하세요. 기존 `triples`를 복사하거나 골드 파일을 읽어 만들지 않습니다.  
- 각 항목에 문자열 **`subject`**, **`relation`**, **`object`**, **`source_doc_id`**, **`evidence`**와 정수 **`sent_id`**를 넣으세요. evidence는 해당 원문 문장 전체를 사용합니다.  
- 지침에 맞는 질병에서 유전자로 가는 관계를 각각 한 항목으로 적으세요. 유전자 이름이 등장해도 그 유전자와 질병의 연관을 진술하지 않으면 제외합니다.  
- **`zero_notes`**를 `{정수 문장번호: 0건으로 판정한 이유 문자열}` 딕셔너리로 작성하세요. 관계가 없는 문장도 읽었다는 기록입니다.  
- **`reviewed_sent_ids`**에 골드를 만든 문장 번호와 0건으로 기록한 문장 번호를 합친 집합을 담으세요. 이 집합이 `mini_plan["sent_ids"]`와 같아야 빠뜨린 문장이 없습니다.

**확인 기준**: 문장 12에서 2건, 문장 13에서 0건입니다. 골드의 이름은 해당 원문에 그대로 있고 0건 문장에는 제외 이유가 있습니다. reviewed_sent_ids는 {12, 13}입니다.

<details><summary>힌트</summary>

**접근방법**: 질병을 주어로 놓고 위험 연관이 명시된 유전자를 하나씩 목적어로 적습니다.

**세부구현**

1. 질병 위험을 높인다고 진술한 유전자와 뒤 절에서 별도 기능을 설명한 유전자를 구분하세요.  
2. 출처 문서와 문장 번호, 원문 전체를 함께 기록하세요.  
3. 다음 문장에 새 관계가 없는 이유를 적고, 두 기록의 문장 번호를 합쳐 검토 범위를 확인하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(mini_gold, list) and len(mini_gold) == 2, "관계마다 한 항목씩 직접 작성하세요."
assert {triple_key(row) for row in mini_gold} == {
    ("Crohn's disease", "ASSOCIATES", "NOD2"), ("Crohn's disease", "ASSOCIATES", "ATG16L1")
}, "질병 위험과 연결된 두 유전자를 확인하세요."
assert all(row["source_doc_id"] == mini_doc_id and row["sent_id"] == 12 for row in mini_gold), "두 항목의 출처는 mini_doc_id이고 sent_id는 정수 12입니다."
assert all(row["evidence"] == mini_sentences[row["sent_id"]] for row in mini_gold), "근거는 원문 문장 전체입니다."
assert set(zero_notes) == {13} and isinstance(zero_notes[13], str) and zero_notes[13].strip(),             "zero_notes의 키는 문자열이 아니라 정수 13이고, 값은 비어 있지 않은 이유 문장입니다."
assert reviewed_sent_ids == {12, 13} == set(mini_plan["sent_ids"]), "0건 문장까지 검토 기록에 넣으세요."
print("✅ 골드 형식·검토 범위 통과. 원문을 읽고 직접 작성했는지도 확인하세요.")

## 8. 직접 만든 골드로 TP, FP, FN을 세세요

**배경**: 사람의 판정 정밀도와 별도로, 고정 범위의 고유 관계를 골드와 비교합니다.

**요구사항**

- **`mini_predictions`**를 `valid` 중 `source_doc_id`가 mini_doc_id이고 `relation`이 `"ASSOCIATES"`인 행만 남긴 원래 순서의 리스트로 만드세요.  
- **`in_annotation_scope(row)`** 함수를 만드세요. 그 행의 주어와 목적어가 **둘 다** `mini_sentences[12]` 안에 있으면 True, 하나라도 없으면 False인 불리언을 돌려줍니다.  
- **`mini_scope_ok`**에는 mini_predictions의 모든 행이 그 함수를 통과하는지 확인한 불리언을 담으세요. 이 저장 추출본에서 두 관계의 근거는 모두 주석한 문장 12에 있습니다.  
- **`mini_pred_keys`**, **`mini_gold_keys`**를 각각 mini_predictions와 mini_gold의 triple_key 집합으로 만드세요.  
- 정수 **`mini_tp`**, **`mini_fp`**, **`mini_fn`**에 교집합의 크기, 추출에만 있는 차집합의 크기, 골드에만 있는 차집합의 크기를 담으세요.  
- 세 값을 출력하세요.

**확인 기준**: in_annotation_scope는 주어와 목적어 중 하나만 벗어나도 False입니다. mini_scope_ok는 True이고 mini_tp=2, mini_fp=0, mini_fn=0입니다. 이 두 문장 점수를 여섯 문서 전체의 품질로 확대하지 않습니다.

<details><summary>힌트</summary>

**접근방법**: 평가 범위를 먼저 맞춘 뒤 집합 연산으로 세 값을 셉니다.

**세부구현**

1. 출처와 관계를 함께 조건으로 사용하세요.  
2. 주어와 목적어를 모두 검사해야 하나만 벗어난 행도 걸러집니다.  
3. 차집합의 방향을 구분하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert mini_predictions == [row for row in valid if row["source_doc_id"] == mini_doc_id and row["relation"] == "ASSOCIATES"], "문서와 관계를 함께 조건으로 걸러 원래 순서를 유지하세요."
assert in_annotation_scope(mini_predictions[0]) is True, "두 이름이 모두 문장 12에 있으면 True입니다."
assert in_annotation_scope(dict(mini_predictions[0], object="문장 12에 없는 목적어")) is False,             "주어뿐 아니라 목적어도 문장 안에 있는지 확인하세요."
assert in_annotation_scope(dict(mini_predictions[0], subject="문장 12에 없는 주어")) is False,             "목적어뿐 아니라 주어도 문장 안에 있는지 확인하세요."
assert mini_scope_ok is True, "선택된 관계가 모두 주석 범위 안에 있는지 확인하세요."
assert mini_pred_keys == {triple_key(row) for row in mini_predictions}, "mini_predictions의 triple_key 집합을 담으세요."
assert mini_gold_keys == {triple_key(row) for row in mini_gold}, "mini_gold의 triple_key 집합을 담으세요."
assert (mini_tp, mini_fp, mini_fn) == (2, 0, 0), "교집합 2건, 추출에만 0건, 골드에만 0건입니다. 차집합의 방향을 확인하세요."
print("✅ TP·FP·FN 통과!")

## 9. 공식을 그대로 적용해 정밀도, 재현율, F1을 구하세요

**배경**: 세 비율은 같은 TP를 서로 다른 분모로 나눈 값입니다. 제공 함수를 쓰기 전에 공식을 직접 확인합니다.

**요구사항**

- **`prf(tp, fp, fn)`** 함수를 만드세요. 세 정수를 받아 `(정밀도, 재현율, F1)` 튜플을 돌려줍니다.  
- 정밀도는 `tp / (tp + fp)`, 재현율은 `tp / (tp + fn)`, F1은 `2 * tp / (2 * tp + fp + fn)`입니다. 분모가 0인 입력은 주지 않습니다.  
- **`mini_prf`**에 앞 문제의 mini_tp, mini_fp, mini_fn을 넣은 결과를 담으세요.  
- **`mini_metrics`**에 제공 함수 `measure_exact`로 mini_predictions와 mini_gold를 비교한 딕셔너리를 담으세요.  
- 두 결과가 같은지 확인하는 문장을 출력하세요. 샘플 정밀도를 P 자리에 넣지 않습니다.

**확인 기준**: mini_prf는 (1.0, 1.0, 1.0)이고 mini_metrics의 precision, recall, f1과 같습니다. 다른 값을 넣어도 공식이 맞아야 하므로 자가채점은 (3, 1, 2) 같은 입력도 확인합니다.

<details><summary>힌트</summary>

**접근방법**: 공식 세 개를 함수 하나에 담고 제공 함수의 결과와 대조합니다.

**세부구현**

1. 분모가 서로 다르다는 점을 확인하고 순서대로 계산하세요.  
2. F1은 정밀도와 재현율을 다시 계산하지 않고 TP, FP, FN으로 바로 구할 수 있습니다.  
3. 제공 함수의 precision, recall, f1 키와 비교하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(prf(3, 1, 2), tuple), "prf는 리스트가 아니라 튜플을 돌려줍니다."
assert prf(3, 1, 2) == (0.75, 0.6, 2 / 3), "세 공식의 분모를 확인하세요."
assert prf(1, 3, 4) == (0.25, 0.2, 2 / 9), "다른 값에서도 세 공식이 맞아야 합니다."
assert mini_prf == prf(mini_tp, mini_fp, mini_fn) == (1.0, 1.0, 1.0), "앞 문제에서 센 세 값을 그대로 넣으세요."
assert mini_metrics == measure_exact(mini_predictions, mini_gold), "제공 함수에 mini_predictions와 mini_gold를 넣으세요."
assert mini_prf == (mini_metrics["precision"], mini_metrics["recall"], mini_metrics["f1"]), "직접 계산한 세 값과 제공 함수의 세 값이 같아야 합니다."
print("✅ 정밀도·재현율·F1 통과!")

## 10. 지표가 낮을 때 어디를 읽을지 정하세요

**배경**: 네 지표는 검사 대상과 분모가 서로 다릅니다. 그래서 낮아졌을 때 돌아가 읽을 목록도 서로 다릅니다.

**요구사항**

- **확인 계획**을 네 지표마다 한 줄씩 적으세요. 스키마 준수율, 근거 원문 일치율, 샘플 정밀도, 골드 완전일치입니다.  
- 각 줄에 그 지표가 낮을 때 **어떤 목록을 읽는지**와 이번 자료에서 그 목록이 몇 행인지 적으세요.  
- 골드 완전일치는 정밀도가 낮을 때와 재현율이 낮을 때 돌아갈 곳이 다릅니다. 둘을 구분해 적으세요.  
- 네 값을 평균 내어 한 숫자로 합치면 안 되는 이유를 한 문장으로 적으세요.

**확인 기준**: 지표마다 읽을 목록이 다르고, 이번 자료의 실제 행 수를 근거로 씁니다. 자동 채점은 하지 않습니다. 작성한 뒤 정답 노트북의 모범 서술과 비교하세요.

<details><summary>힌트</summary>

**접근방법**: 지표마다 분모가 무엇이었는지 되짚고, 그 분모가 가리키는 목록을 적으세요.

**세부구현**

1. 각 지표의 분모에 어떤 행이 들어갔는지 먼저 적으세요.  
2. 그 행들 중 문제가 있는 쪽만 모은 목록의 이름과 행 수를 적으세요.  
3. 정밀도와 재현율은 분모가 다르므로 돌아갈 곳도 나누어 적으세요.

</details>

*(여기에 원문 근거와 판단을 작성하세요.)*

## 11. 이번 평가 결과를 파일로 남기세요

**배경**: 서로 다른 검사와 평가 범위를 한 숫자로 합치지 않고 기록합니다.

**요구사항**

- **`report`**를 딕셔너리로 만드세요. **`schema`**에는 schema_report, **`evidence`**에는 evidence_report를 넣습니다.  
- **`judged`**에는 딕셔너리 `{"population_rows": evidence_ok의 행 수, "rate": precision_all, "sample_rows": sample의 행 수, "sample_rate": sample_precision}`을 넣습니다.  
- **`mini_scope`**, **`mini_exact`**에는 mini_plan, mini_metrics를 각각 넣습니다.  
- **`next_reads`**에는 딕셔너리 `{"schema_rejected": rejected의 행 수, "evidence_review": evidence_review의 행 수, "judged_zero": sample 중 채점표에서 0점인 행 수, "exact_fp_fn": mini_metrics의 fp와 fn을 더한 값}`을 넣습니다. 지표가 낮을 때 읽어야 할 목록의 크기입니다.  
- 제공된 저장 함수로 report를 **`lv1_report.json`**, mini_gold를 **`lv1_mini_gold.json`**, zero_notes를 **`lv1_zero_notes.json`**으로 저장하세요. 세 파일은 output_dir에 생깁니다.

**확인 기준**: 세 파일이 생성되고 다시 읽은 report가 원래 report와 같습니다. 스키마 준수율, 근거 원문 일치율, 샘플 정밀도, 완전일치 점수가 각각 별도 항목이고 next_reads는 3, 6, 2, 0입니다.

<details><summary>힌트</summary>

**접근방법**: 이미 계산한 값에 평가 범위를 붙여 저장합니다.

**세부구현**

1. 지표를 반올림하지 않은 값으로 보관하세요.  
2. 0점 행은 채점표에서 값이 0인 표본 행을 세면 됩니다.  
3. write_json으로 각각 저장하세요.

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert report["schema"] == schema_report and report["evidence"] == evidence_report,             "앞에서 만든 두 딕셔너리를 그대로 넣으세요."
assert report["judged"] == {"population_rows": len(evidence_ok), "rate": precision_all,
                            "sample_rows": len(sample), "sample_rate": sample_precision},             "네 키의 이름을 확인하고, 비율은 반올림하지 말고 계산한 값을 그대로 넣으세요."
assert report["mini_scope"] == mini_plan and report["mini_exact"] == mini_metrics,             "mini_scope에는 mini_plan을, mini_exact에는 mini_metrics를 넣으세요."
assert report["next_reads"] == {"schema_rejected": len(rejected), "evidence_review": len(evidence_review),
                                "judged_zero": sum(1 for row in sample if scores[triple_key(row)] == 0),
                                "exact_fp_fn": mini_metrics["fp"] + mini_metrics["fn"]},             "네 키의 이름과 값을 확인하세요. 지표가 아니라 읽을 목록의 행 수를 담습니다."
assert list(report["next_reads"].values()) == [3, 6, 2, 0], "기각 3행, 검토 6행, 0점 2행, FP와 FN 합 0입니다."
assert json.loads((output_dir / "lv1_report.json").read_text()) == report, "파일 내용까지 확인하세요."
assert json.loads((output_dir / "lv1_mini_gold.json").read_text()) == mini_gold, "mini_gold를 lv1_mini_gold.json으로 저장하세요."
assert json.loads((output_dir / "lv1_zero_notes.json").read_text()) == {str(k): v for k, v in zero_notes.items()}, "zero_notes를 lv1_zero_notes.json으로 저장하세요. JSON은 키를 문자열로 바꿔 적습니다."
print("✅ LV1 결과 저장 통과!")